# 🧬 Project 5 — Semantic Correspondence (Extended Version)
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

Pipeline avanzata con analisi multi-backbone, PEFT, raffinamento adattivo e test di generalizzazione su nuovi domini e compiti geometrici.

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

import os
from utils.demo_utils import launch_stage_demo, launch_comparison_demo

## 🔍 1. Stage 1: Backbone Analysis (Baseline & Multi-Threshold)
Analisi comparativa seguendo le soglie PCK@0.05, 0.1 e 0.2.

In [ ]:
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2_extended.txt

In [ ]:
# Esplorazione dei layer
launch_stage_demo('DINOv2 Layer Explorer', show_layer_slider=True)

## 🚀 2. Stage 2: Fine-Tuning Efficiente (PEFT)
Addestramento LoRA e BitFit con logica di persistenza.

In [ ]:
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
if not os.path.exists(f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir {DRIVE_CKPTS}/lora_only
else:
    print('[INFO] LoRA già addestrato. Salto training.')

## 🎯 3. Stage 3: Raffinamento e Risultati Finali
Valutazione del modello LoRA + Adaptive Window.

In [ ]:
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
if os.path.exists(CKPT_LORA):
    !python evaluate.py --dataset_root ./data/SPair-71k --checkpoint {CKPT_LORA} --results_file /content/drive/MyDrive/AML/Results/final_lora_aw.txt
    launch_stage_demo('LoRA + Adaptive Window', ckpt_name='lora_only')
else:
    print('[ERROR] Checkout LoRA non trovato. Lanciare lo Stage 2.')

## 📐 4. Stage 4: Geometric Tasks & Resilience
Valutazione sulla robustezza geometrica.

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

## 🏆 5. Comparison Showcase (Baseline vs Optimized)

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')